## Budget kill switch

Runs every 3 hours as a scheduled job. It reads the spending from the billing system table, and if the
spending of the current month or of today exceeds its limit, it:

1. pauses the schedule of every other job,
2. cancels their active and queued runs,
3. stops every SQL warehouse and every app,
4. then fails on purpose, so the job's failure notification sends you an email.

A dev run (the interactive default) only reports what a production run would do, and changes nothing.

To resume after a stop: raise the limit (or wait for the new day or month), then unpause the job schedules by hand.

The billing table is a few hours behind, so the switch reacts within hours, not minutes.
Costs are computed at list prices, which can differ from what the credits are charged.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import JobSettings, PauseStatus

# Scheduled runs pass run_mode = "production". Interactive runs default to "dev",
# which reports what the switch would do and changes nothing.
dbutils.widgets.dropdown("run_mode", "dev", ["dev", "production"])
IS_PRODUCTION = dbutils.widgets.get("run_mode") == "production"

MONTHLY_LIMIT_USD = 900        # 20% above the $750 monthly plan
DAILY_LIMIT_USD = 0           # twice the $25 daily warning: a runaway, not a busy day
KILL_SWITCH_JOB_NAME = "budget-kill-switch"   # this job; it is never paused by itself

spending = spark.sql("""
    SELECT
      SUM(u.usage_quantity * p.pricing.default) AS month_to_date_usd,
      SUM(CASE WHEN u.usage_date = current_date() THEN u.usage_quantity * p.pricing.default ELSE 0 END) AS today_usd
    FROM system.billing.usage u
    JOIN system.billing.list_prices p
      ON u.sku_name = p.sku_name
     AND u.usage_start_time >= p.price_start_time
     AND (p.price_end_time IS NULL OR u.usage_start_time < p.price_end_time)
    WHERE u.usage_date >= date_trunc('month', current_date())
""").first()

month_to_date_usd = spending.month_to_date_usd or 0.0
today_usd = spending.today_usd or 0.0
print(f"Run mode: {'production' if IS_PRODUCTION else 'dev'}")
print(f"Month to date: ${month_to_date_usd:,.2f} of ${MONTHLY_LIMIT_USD:,} | Today: ${today_usd:,.2f} of ${DAILY_LIMIT_USD:,}")

over_budget = month_to_date_usd > MONTHLY_LIMIT_USD or today_usd > DAILY_LIMIT_USD

In [0]:
# Stop everything that spends money, if a limit is exceeded.

if over_budget and not IS_PRODUCTION:
    print("dev run: a limit is exceeded; a production run would pause the jobs and stop the warehouses and apps")
elif over_budget:
    workspace = WorkspaceClient()
    paused_jobs = []

    for job in workspace.jobs.list():
        job_settings = workspace.jobs.get(job.job_id).settings
        if job_settings.name == KILL_SWITCH_JOB_NAME:
            continue
        # Jobs can be scheduled two ways: a cron schedule, or a trigger (periodic, file arrival, table update)
        if job_settings.schedule and job_settings.schedule.pause_status != PauseStatus.PAUSED:
            job_settings.schedule.pause_status = PauseStatus.PAUSED
            workspace.jobs.update(job_id=job.job_id, new_settings=JobSettings(schedule=job_settings.schedule))
            paused_jobs.append(job_settings.name)
        if job_settings.trigger and job_settings.trigger.pause_status != PauseStatus.PAUSED:
            job_settings.trigger.pause_status = PauseStatus.PAUSED
            workspace.jobs.update(job_id=job.job_id, new_settings=JobSettings(trigger=job_settings.trigger))
            paused_jobs.append(job_settings.name)
        workspace.jobs.cancel_all_runs(job_id=job.job_id)

    for warehouse in workspace.warehouses.list():
        workspace.warehouses.stop(warehouse.id)

    for app in workspace.apps.list():
        workspace.apps.stop(app.name)

    # Fail on purpose, so the job's failure notification sends an email
    raise RuntimeError(
        f"Budget limit exceeded (month to date ${month_to_date_usd:,.2f}, today ${today_usd:,.2f}). "
        f"Paused schedules: {paused_jobs}. Stopped all SQL warehouses and apps."
    )
else:
    print("Within budget; nothing changed.")